In [ ]:
!ls -R

In [ ]:
import polars as pl

df = pl.read_parquet( 'data.parquet' )


In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns)
print("Dtypes:", df.dtypes)

#df.describe()

In [ ]:
df

In [ ]:
import polars as pl

# Load your Parquet file
df = pl.read_parquet("data.parquet")

print("=== BASIC INFO ===")
print("Shape:", df.shape)
print("Columns:", df.columns)
print("Dtypes:", df.dtypes)
print()

# -----------------------------------------
# 1. Numerical summary
# -----------------------------------------
print("=== NUMERICAL SUMMARY ===")
numeric_cols = [
    col for col, dtype in zip(df.columns, df.dtypes)
    if dtype in (pl.Int64, pl.Float64)
]

print(df.select(numeric_cols).describe())
print()

# -----------------------------------------
# 2. Null counts
# -----------------------------------------
print("=== NULL COUNTS PER COLUMN ===")
print(df.null_count())
print()

# -----------------------------------------
# 3. Category frequencies (top 10)
# -----------------------------------------
print("=== TOP CATEGORIES ===")

categorical_cols = [
    col for col, dtype in zip(df.columns, df.dtypes)
    if dtype == pl.String
]

for col in categorical_cols:
    print(f"\n-- {col} (top 10) --")
    print(
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
    )

print()

# -----------------------------------------
# 4. Correlation matrix
# -----------------------------------------
print("=== CORRELATION MATRIX (NUMERIC) ===")
corr = df.select(numeric_cols).corr()
print(corr)
print()

# -----------------------------------------
# 5. Outlier inspection for key numeric columns
# -----------------------------------------
print("=== OUTLIER CANDIDATE CHECK ===")

key_numeric = ["ARQ_TAM", "FALTA_TAM", "DEPTH"]

existing = [col for col in key_numeric if col in df.columns]

print(
    df.select([
        pl.col(col).min().alias(f"{col}_min")
        for col in existing
    ] + [
        pl.col(col).max().alias(f"{col}_max")
        for col in existing
    ] + [
        pl.col(col).quantile(0.95).alias(f"{col}_p95")
        for col in existing
    ])
)
print()

# -----------------------------------------
# 6. Group-by insights
# -----------------------------------------
print("=== GROUP-BY INSIGHTS ===")

# Example: mean DEPTH per OIL_RIG

# Mean DEPTH per OIL_RIG
if "OIL_RIG" in df.columns and "DEPTH" in df.columns:
    print("\n-- Mean DEPTH per OIL_RIG (top 15 deepest) --")
    print(
        df.group_by("OIL_RIG")
          .agg(pl.col("DEPTH").mean().alias("DEPTH_mean"))   # FIXED
          .sort("DEPTH_mean", descending=True)               # FIXED
          .head(15)
    )
# Total FALTA_TAM per COMPANY
if "COMPANY" in df.columns and "FALTA_TAM" in df.columns:
    print("\n-- Total FALTA_TAM per COMPANY (top 15) --")
    print(
        df.group_by("COMPANY")
          .agg(pl.col("FALTA_TAM").sum().alias("FALTA_TAM_sum"))   # FIXED
          .sort("FALTA_TAM_sum", descending=True)                  # FIXED
          .head(15)
    )

print()

# -----------------------------------------
# 7. Lazy-mode summary (fast for larger data)
# -----------------------------------------
print("=== LAZY SUMMARY (FAST OVERVIEW) ===")

df_lazy = pl.scan_parquet("data.parquet")

lazy_summary = df_lazy.describe()
print(lazy_summary)

print("\n=== EDA COMPLETE ===")

In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pl.read_parquet("data.parquet")

# Convert to pandas only when needed for plotting
pdf = df.to_pandas()

# -----------------------------
# 1. Numerical Histograms
# -----------------------------
numeric_cols = [
    c for c, t in zip(df.columns, df.dtypes)
    if t in (pl.Int64, pl.Float64)
]

for col in numeric_cols:
    fig = px.histogram(pdf, x=col, nbins=40, title=f"Histogram: {col}")
    fig.show()

# -----------------------------
# 2. Boxplots for numeric columns
# -----------------------------
for col in numeric_cols:
    fig = px.box(pdf, y=col, title=f"Boxplot: {col}")
    fig.show()

# -----------------------------
# 3. Correlation heatmap
# -----------------------------
corr = pdf[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=False, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# -----------------------------
# 4. Top categories (bar charts)
# -----------------------------
categorical_cols = [
    c for c, t in zip(df.columns, df.dtypes)
    if t == pl.String
]

for col in categorical_cols:
    top10 = (
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
          .to_pandas()
    )
    fig = px.bar(top10, x=col, y="len", title=f"Top 10: {col}")
    fig.show()

In [ ]:
# ============================================
# 1. LOAD DATA
# ============================================
import polars as pl

df = pl.read_parquet("data.parquet")

# --------------------------------------------
# SEMANTIC COLUMN GROUPING (based on your explanation)
# --------------------------------------------
id_cols = ["ID"]

categorical_cols = [
    "SERVICE", "COMPANY", "TYPE", "NAME",
    "BUCKET", "BASE", "GROUP", "UN",
    "FIELD", "OIL_RIG"
]

numeric_existing_cols = ["ARQ_QTD", "ARQ_TAM"]
numeric_missing_cols = ["FALTA_QTD", "FALTA_TAM"]
numeric_depth = ["DEPTH"]

# OLDEST, NEWEST, DATE are DATE columns (FIXED)
date_cols = ["OLDEST", "NEWEST", "DATE"]

all_numeric = numeric_existing_cols + numeric_missing_cols + numeric_depth

# ============================================
# 2. PARSE DATE COLUMNS (FIXED)
# ============================================
for col in date_cols:
    df = df.with_columns(
        pl.col(col).str.strptime(pl.Date, strict=False)
    )

# ============================================
# 3. ENCODE CATEGORICAL COLUMNS
# ============================================
df = df.with_columns([
    pl.col(col).cast(pl.Categorical) for col in categorical_cols
])

# integer-encoded version for correlation
df_encoded = df.with_columns([
    pl.col(col).to_physical().alias(col + "_ENC") for col in categorical_cols
])

# ============================================
# 4. TEXTUAL EDA
# ============================================

print("=== BASIC INFO ===")
print("Shape:", df.shape)
print("Columns:", df.columns)
print("Dtypes:", df.dtypes)

# Numerical summary
print("\n=== NUMERICAL SUMMARY ===")
print(df.select(all_numeric).describe())

# Missing values
print("\n=== MISSING VALUES ===")
print(df.null_count())

# Top categories
print("\n=== TOP 10 CATEGORY VALUES ===")
for col in categorical_cols:
    print(f"\nTop 10 for {col}:")
    print(
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
    )

# ============================================
# 5. CORRELATION ANALYSIS (with encoded categories)
# ============================================
print("\n=== CORRELATION MATRIX ===")

numeric_for_corr = all_numeric + [c + "_ENC" for c in categorical_cols]

pdf_corr = df_encoded.select(numeric_for_corr).to_pandas()
corr_matrix = pdf_corr.corr()
print(corr_matrix)

# ============================================
# 6. VISUAL EDA (Plotly + Matplotlib)
# ============================================
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

pdf = df_encoded.to_pandas()

# Histograms
for col in all_numeric:
    fig = px.histogram(pdf, x=col, nbins=40, title=f"Histogram: {col}")
    fig.show()

# Boxplots
for col in all_numeric:
    fig = px.box(pdf, y=col, title=f"Boxplot: {col}")
    fig.show()

# Correlation Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, cmap="coolwarm")
plt.title("Correlation Heatmap (Including Encoded Categorical Columns)")
plt.show()

# Top category bars
for col in categorical_cols:
    top10 = (
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
          .to_pandas()
    )
    fig = px.bar(top10, x=col, y="len", title=f"Top 10 categories for {col}")
    fig.show()

# ============================================
# 7. PROFILING REPORT (Polars → Pandas)
# ============================================
from ydata_profiling import ProfileReport

profile = ProfileReport(
    pdf,
    title="Full Dataset Profiling Report",
    explorative=True
)

profile.to_file("profiling_report.html")
print("\nProfiling saved as profiling_report.html")

print("\n=== EDA COMPLETE ===")

In [ ]:
# ============================================
# FULL SERVICE/VESSEL EDA PIPELINE (ONE CELL)
# ============================================

# Install dependencies if needed (Jupyter-compatible)
import sys
!{sys.executable} -m pip install polars plotly seaborn matplotlib ydata-profiling --quiet

# Imports
import polars as pl
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
#from ydata_profiling import ProfileReport

# ============================================
# 1. LOAD DATA
# ============================================

df = pl.read_parquet("data.parquet")

categorical_cols = [
    "SERVICE", "COMPANY", "TYPE", "NAME",
    "BUCKET", "BASE", "GROUP", "UN",
    "FIELD", "OIL_RIG"
]

numeric_existing = ["ARQ_QTD", "ARQ_TAM"]
numeric_missing = ["FALTA_QTD", "FALTA_TAM"]
numeric_depth = ["DEPTH"]

date_cols = ["OLDEST", "NEWEST", "DATE"]

all_numeric = numeric_existing + numeric_missing + numeric_depth

# ============================================
# 2. PARSE DATE COLUMNS
# ============================================

for col in date_cols:
    df = df.with_columns(pl.col(col).str.strptime(pl.Date, strict=False))

# ============================================
# 3. ENCODE CATEGORICAL COLUMNS
# ============================================

df = df.with_columns([
    pl.col(col).cast(pl.Categorical) for col in categorical_cols
])

df_encoded = df.with_columns([
    pl.col(col).to_physical().alias(col + "_ENC") for col in categorical_cols
])

# ============================================
# 4. BASIC TEXT EDA
# ============================================

print("Shape:", df.shape)
print("Columns:", df.columns)
print("Dtypes:", df.dtypes)

print("\nNUMERIC SUMMARY")
print(df.select(all_numeric).describe())

print("\nNULL COUNTS")
print(df.null_count())

# Top categories
print("\n=== TOP 10 CATEGORIES ===")
for col in categorical_cols:
    print(f"\nTop 10 for {col}:")
    print(
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
    )

# ============================================
# 5. CORRELATION MATRIX (ENCODED)
# ============================================

numeric_for_corr = all_numeric + [c + "_ENC" for c in categorical_cols]
pdf_corr = df_encoded.select(numeric_for_corr).to_pandas()
corr_matrix = pdf_corr.corr()

print("\nCorrelation matrix calculated.")

# ============================================
# 6. VISUAL EDA
# ============================================

pdf = df_encoded.to_pandas()

# Histograms
for col in all_numeric:
    fig = px.histogram(pdf, x=col, nbins=40, title=f"Histogram: {col}")
    fig.show()

# Boxplots
for col in all_numeric:
    fig = px.box(pdf, y=col, title=f"Boxplot: {col}")
    fig.show()

# Heatmap
plt.figure(figsize=(14,10))
sns.heatmap(corr_matrix, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# Top categories bar charts
for col in categorical_cols:
    top10 = (
        df.group_by(col).len().sort("len", descending=True).head(10).to_pandas()
    )
    fig = px.bar(top10, x=col, y="len", title=f"Top 10 categories for {col}")
    fig.show()

# ============================================
# 7. PROFILING REPORT
# ============================================

#profile = ProfileReport(pdf, title="Dataset Profiling Report", explorative=True)
#profile.to_file("profiling_report.html")

#print("\nSaved profiling_report.html")
print("\n=== EDA COMPLETE ===")

In [ ]:
import polars as pl

# ============================================
# Define schema based on your description
# ============================================
schema = {
    "ID": pl.Int64,
    "SERVICE": pl.Float64,
    "COMPANY": pl.Utf8,
    "TYPE": pl.Utf8,
    "NAME": pl.Utf8,
  "ARQ_QTD": pl.Int64,
    "ARQ_TAM": pl.Float64,
    "FALTA_QTD": pl.Int64,
    "FALTA_TAM": pl.Float64,
  "OLDEST": pl.Utf8,   # will be converted to date after load
    "NEWEST": pl.Utf8,   # will be converted to date after load
  "BUCKET": pl.Utf8,
    "BASE": pl.Utf8,
    "GROUP": pl.Utf8,
  "DATE": pl.Utf8,     # also converted after load
  "UN": pl.Utf8,
    "FIELD": pl.Float64,
    "OIL_RIG": pl.Utf8,
    "DEPTH": pl.Float64,
}

# ============================================
# Load CSV with forced schema
# ============================================
df = pl.read_csv(
    "pacotes.csv",
    schema=schema,
    ignore_errors=True # avoids crash if some rows have issues
)

# ============================================
# Convert string → date columns
# ============================================

df = df.with_columns([
    pl.col("OLDEST").str.strptime(pl.Date, strict=False),
    pl.col("NEWEST").str.strptime(pl.Date, strict=False),
    pl.col("DATE").str.strptime(pl.Date, strict=False)
])

# ============================================
# Save as Parquet
# ============================================
df.write_parquet("data-02.parquet")

print("CSV successfully converted to Parquet with correct data types!")
print(df.head())
print(df.dtypes)

In [ ]:
# ============================================================
# FULL UPDATED EDA PIPELINE USING CORRECTED COLUMN TYPES
# ============================================================

# Install required libs if needed
import sys
!{sys.executable} -m pip install polars plotly seaborn matplotlib ydata-profiling --quiet

# Imports
import polars as pl
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
#from ydata_profiling import ProfileReport

# ============================================================
# 1. LOAD FINAL PARQUET
# ============================================================

df = pl.read_parquet("data.parquet")

# ============================================================
# 2. COLUMN GROUPING (BASED ON VERIFIED TYPES)
# ============================================================

categorical_cols = [
    "COMPANY", "TYPE", "NAME",
    "BUCKET", "BASE", "GROUP", "UN",
    "OIL_RIG"
]

# SERVICE order sometimes float/int — keep numeric
numeric_cols = [
    "ID", "SERVICE",
    "ARQ_QTD", "ARQ_TAM",
    "FALTA_QTD", "FALTA_TAM",
    "FIELD", "DEPTH"
]

date_cols = ["OLDEST", "NEWEST", "DATE"]

# ============================================================
# 3. ENSURE DATE COLUMNS ARE PARSED
# ============================================================

for col in date_cols:
    df = df.with_columns(pl.col(col).cast(pl.Date))

# ============================================================
# 4. ENCODE CATEGORICAL COLUMNS
# ============================================================

df = df.with_columns([
    pl.col(col).cast(pl.Categorical) for col in categorical_cols
])

df_encoded = df.with_columns([
    pl.col(col).to_physical().alias(col + "_ENC") for col in categorical_cols
])

# ============================================================
# 5. TEXT EDA
# ============================================================

print("\n=== BASIC INFORMATION ===")
print("Shape:", df.shape)
print("Columns:", df.columns)
print("Dtypes:", df.dtypes)

print("\n=== NUMERIC SUMMARY ===")
print(df.select(numeric_cols).describe())

print("\n=== NULL COUNTS ===")
print(df.null_count())

print("\n=== TOP 10 CATEGORIES ===")
for col in categorical_cols:
    print(f"\nTop 10 for {col}:")
    print(
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
    )

# ============================================================
# 6. CORRELATION MATRIX (Numeric + Encoded Categorical)
# ============================================================

numeric_for_corr = numeric_cols + [c + "_ENC" for c in categorical_cols]
pdf_corr = df_encoded.select(numeric_for_corr).to_pandas()
corr_matrix = pdf_corr.corr()

print("\nCorrelation matrix:")
print(corr_matrix)

# ============================================================
# 7. VISUAL EDA
# ============================================================

pdf = df_encoded.to_pandas()

# Histograms
for col in numeric_cols:
    fig = px.histogram(pdf, x=col, nbins=40, title=f"Histogram: {col}")
    fig.show()

# Boxplots
for col in numeric_cols:
    fig = px.box(pdf, y=col, title=f"Boxplot: {col}")
    fig.show()

# Correlation Heatmap
plt.figure(figsize=(14,10))
sns.heatmap(corr_matrix, cmap="coolwarm")
plt.title("Correlation Heatmap (with Encoded Categories)")
plt.show()

# Category frequencies
for col in categorical_cols:
    top10 = (
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .head(10)
          .to_pandas()
    )
    fig = px.bar(top10, x=col, y="len", title=f"Top 10 {col}")
    fig.show()

# ============================================================
# 8. PROFILING REPORT
# ============================================================

#profile = ProfileReport(pdf, title="Dataset Profiling Report", explorative=True)
#profile.to_file("profiling_report.html")

print("\n✔ EDA Complete — profiling_report.html generated.")